In [1]:
import torch
import pickle
import time
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import faiss
import ujson as json
from pathlib import Path
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


In [2]:
from backend.rag_core import rag_pipeline

Loaded index with 5077 vectors; metadata rows: 5077


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [3]:
DATASET_NAME = "my_rag_eval_set"

In [7]:
from datetime import datetime

def retriever_log(dataset, k=10):
    run_log = []
    for ex in dataset:
        q = ex["question"]
        gold = ex.get("answer")          # None if not available
        qid = ex.get("id")               # optional

        out = rag_pipeline(q, k=k)

        run_log.append({
            # identifiers
            "query_id": qid,
            "dataset": DATASET_NAME,

            # I/O
            "query": q,
            "gold_answer": gold,
            "answer": out["answer"],

            # retrieval output
            "contexts": out["contexts"],      # you’ll use this for Recall@k etc.

            # performance
            "timing": out["timing"],          # p50/p95 latency later

            # experiment meta
            "rag_config": out["config"],
            "timestamp": datetime.utcnow().isoformat(),
        })

    return run_log

In [8]:
def save_log(run_log, path: Path):
    with path.open("w", encoding="utf-8") as f:
        for row in run_log:
            f.write(json.dumps(row) + "\n")

In [10]:
# Example toy dataset
dataset = [
    {"id": 1, "question": "What is GraphRAG?", "answer": "..."},  
]

run_log = retriever_log(dataset, k=10)
save_log(run_log, Path("run_log.graph_rag_v1.jsonl"
))